## 1-setup and loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

df = pd.read_csv("data/raw/imdb_50k.csv")
df["label"] = (df["sentiment"] == "positive").astype(int)
df["n_words"] = df["review"].str.split().str.len()
pl = pd.read_csv("data/raw/pang_lee_2000.csv")
own = pd.read_csv("data/own/own_reviews.csv")
print(df.shape, pl.shape, own.shape)

## 2-structure, balance and duplicates

In [ ]:
print(df.isnull().sum().to_dict())
print(df["label"].value_counts().to_dict())
dup = df[df.duplicated(subset="review")]
print(len(dup), dup["label"].value_counts().to_dict())
print(df[df.duplicated(subset="review", keep=False)].groupby("review")["label"].nunique().gt(1).sum())

## 3-review length

In [ ]:
print(df.groupby("label")["n_words"].describe().round(1))
print(df["n_words"].quantile([0.95, 0.99]).tolist())

## 4-noise and negation 

In [ ]:
print(df["review"].str.contains("<br />").mean())
print(df["review"].apply(lambda s: any(ord(c) > 127 for c in s)).mean())
print(df["review"].str.lower().str.contains(r"\b(?:not|no|never|n't)\b|n't").mean())

## 5-figure 1

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.countplot(x="sentiment", data=df, ax=ax[0])
sns.histplot(data=df, x="n_words", hue="sentiment", bins=60, ax=ax[1])
ax[1].set_xlim(0, 1000)
sns.boxplot(x="sentiment", y="n_words", data=df, ax=ax[2])
ax[2].set_ylim(0, 800)
plt.tight_layout()
plt.savefig("figures/01_balance_length.png", dpi=150)
plt.show()

## 6-frequent words and bigrams

In [ ]:
txt = df["review"].str.replace(r"<.*?>", " ", regex=True)

def top_ngrams(texts, ngram, n=12):
    cv = CountVectorizer(stop_words="english", ngram_range=ngram)
    X = cv.fit_transform(texts)
    freq = np.asarray(X.sum(axis=0)).ravel()
    vocab = cv.get_feature_names_out()
    return [(vocab[i], int(freq[i])) for i in np.argsort(-freq)[:n]]

for lab in (0, 1):
    sub = txt[df["label"] == lab]
    print(lab, top_ngrams(sub, (1, 1)))
    print(lab, top_ngrams(sub, (2, 2)))

## 7-discriminative words and fig 2

In [ ]:
tv = TfidfVectorizer(stop_words="english", max_features=20000, min_df=5)
X = tv.fit_transform(txt)
lr = LogisticRegression(max_iter=1000).fit(X, df["label"])
vocab = tv.get_feature_names_out()
order = np.argsort(lr.coef_[0])
idx = list(order[:15]) + list(order[-15:])

plt.figure(figsize=(7, 7))
plt.barh(vocab[idx], lr.coef_[0][idx], color=["tab:red"] * 15 + ["tab:green"] * 15)
plt.tight_layout()
plt.savefig("figures/02_top_words.png", dpi=150)
plt.show()

## 8-read 20 reviews by hand


In [ ]:
for _, r in df.sample(20, random_state=1).iterrows():
    print(r["label"], "|", r["review"][:300], "\n")

## 9-compare the 3 datasets

In [ ]:
def profile(d):
    t = d["review"]
    return {
        "reviews": len(d),
        "pos_share": round(d["label"].mean(), 3),
        "mean_words": round(t.str.split().str.len().mean(), 1),
        "median_words": t.str.split().str.len().median(),
        "html_share": round(t.str.contains("<br />").mean(), 3),
        "negation_share": round(t.str.lower().str.contains(r"\b(?:not|no|never|n't)\b|n't").mean(), 3),
    }

imdb_dedup = df.drop_duplicates(subset="review")
pd.DataFrame({"IMDB": profile(imdb_dedup), "PangLee": profile(pl), "Own": profile(own)}).T

## 10-hard cases

In [ ]:
print(own.groupby("label")["hard_case"].agg(["sum", "mean"]).round(3))
for _, r in own[own["hard_case"] == 1].head(10).iterrows():
    print(r["label"], r["rating10"], "|", r["review"][:250], "\n")